# Imports

In [ ]:
import numpy as np
import pandas as pd
import glob
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA = Path("../data")
RET_COLS = [f"RET_{i}" for i in range(1, 21)]          # RET_1 = most recent, RET_20 = oldest
VOL_COLS = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]

# Load

In [ ]:
x_train_path = str(DATA / "X_train_9xQjqvZ.csv")
y_train_path = str(DATA / "y_train_Ppwhaz8.csv")
x_test_path  = str(DATA / "X_test_1zTtEnD.csv")

X = pd.read_csv(x_train_path)
y = pd.read_csv(y_train_path)
# y.columns = [c.lower() for c in y.columns]              # y_train uses 'target'

df = X.merge(y, on="ROW_ID", how="left", validate="one_to_one")

# downcast floats to halve memory
floatcols = df.select_dtypes("float64").columns
df[floatcols] = df[floatcols].astype("float32")

print(df.shape)
print("merged target nulls:", df["target"].isna().sum())   # should be 0 for train
df.head()

In [ ]:
df["ts_order"]    = df["TS"].str.extract(r"(\d+)").astype(int)
df["alloc_order"] = df["ALLOCATION"].str.extract(r"(\d+)").astype(int)
df = df.sort_values(["ts_order", "alloc_order"]).reset_index(drop=True)

df["target_sign"] = (df["target"] > 0).astype(int)      # the actual label for accuracy
df[["TS", "ts_order", "ALLOCATION", "alloc_order"]].head()

# Exploration

In [ ]:
print(f"rows            : {len(df):,}")
print(f"unique dates    : {df['ts_order'].nunique():,}")
print(f"unique allocs   : {df['alloc_order'].nunique():,}")
print(f"unique groups   : {df['GROUP'].nunique():,}")
print(f"date range      : {df['ts_order'].min()} .. {df['ts_order'].max()}")
print(f"avg allocs/date : {len(df) / df['ts_order'].nunique():.1f}")
print(f"avg dates/alloc : {len(df) / df['alloc_order'].nunique():.1f}")

In [ ]:
per_date  = df.groupby("ts_order").size()
per_alloc = df.groupby("alloc_order").size()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Allocations per date", "Dates per allocation"))
fig.add_trace(go.Scatter(x=per_date.index, y=per_date.values, mode="lines"), row=1, col=1)
fig.add_trace(go.Histogram(x=per_alloc.values, nbinsx=50), row=1, col=2)
fig.update_layout(height=350, showlegend=False, title_text="Panel structure")
fig.show()

In [ ]:
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
if len(miss):
    px.bar(miss, title="% missing by column", labels={"value": "% missing", "index": ""}).show()
else:
    print("No missing values.")

In [ ]:
t = df["target"]
print(t.describe())
print(f"skew={t.skew():.2f}  kurtosis={t.kurtosis():.2f}")

clip = t.quantile([0.005, 0.995]).values          # clip tails just for viewing
px.histogram(t.clip(*clip), nbins=200, title="Target distribution (0.5–99.5% clipped)").show()

In [ ]:
print(f"overall P(target>0) = {df['target_sign'].mean():.4f}")

rate_by_date = df.groupby("ts_order")["target_sign"].mean()
fig = px.scatter(x=rate_by_date.index, y=rate_by_date.values, opacity=0.4,
                 labels={"x": "date order", "y": "P(target>0)"},
                 title="Daily up-rate over time  (look for regime shifts / trend)")
fig.add_hline(y=0.5, line_dash="dash")
fig.add_hline(y=df["target_sign"].mean(), line_color="red")
fig.show()

In [ ]:
g = df.groupby("GROUP").agg(up_rate=("target_sign", "mean"), n=("target_sign", "size")).reset_index()
g = g.sort_values("up_rate")
fig = px.bar(g, x="GROUP", y="up_rate", hover_data=["n"], title="Up-rate by group")
fig.add_hline(y=0.5, line_dash="dash")
fig.show()

In [ ]:
for block, cols in [("RET", RET_COLS), ("SIGNED_VOLUME", VOL_COLS)]:
    print(f"\n=== {block} ===")
    print(df[cols].describe().T[["mean", "std", "min", "50%", "max"]].round(4))

print("\n=== MEDIAN_DAILY_TURNOVER ===")
print(df["MEDIAN_DAILY_TURNOVER"].describe().round(4))

In [ ]:
melt = df[RET_COLS].melt(var_name="lag", value_name="ret")
melt["days_ago"] = melt["lag"].str.extract(r"(\d+)").astype(int)
q = melt.groupby("days_ago")["ret"].agg(["mean", "std",
        lambda s: s.quantile(0.05), lambda s: s.quantile(0.95)])
q.columns = ["mean", "std", "p05", "p95"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=q.index, y=q["mean"], name="mean"))
fig.add_trace(go.Scatter(x=q.index, y=q["std"],  name="std"))
fig.update_layout(title="Return mean & std by days-ago (1 = yesterday)", xaxis_title="days ago")
fig.show()

In [ ]:
rows = []
for i, c in enumerate(RET_COLS, start=1):
    sub = df[[c, "target", "target_sign"]].dropna()
    pear = np.corrcoef(sub[c], sub["target"])[0, 1]                 # continuous corr
    agree = (np.sign(sub[c]) == np.sign(sub["target"])).mean()     # sign agreement
    rows.append({"days_ago": i, "pearson": pear, "sign_agree": agree})
decay = pd.DataFrame(rows)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=decay["days_ago"], y=decay["pearson"], name="corr(RET_i, target)"), secondary_y=False)
fig.add_trace(go.Scatter(x=decay["days_ago"], y=decay["sign_agree"], name="sign agreement"), secondary_y=True)
fig.add_hline(y=0.5, line_dash="dash", secondary_y=True)
fig.update_layout(title="Predictive content by horizon — sign_agree<0.5 ⇒ reversal, >0.5 ⇒ momentum")
fig.show()
decay.round(4)

In [ ]:
def acf(x, nlags):
    x = np.asarray(x, float); x = x - x.mean()
    denom = np.dot(x, x)
    return np.array([1.0 if denom == 0 else np.dot(x[:len(x)-k], x[k:]) / denom
                     for k in range(nlags + 1)])

# RET_1 ordered by date IS each allocation's realized daily return series
piv = df.pivot_table(index="ts_order", columns="alloc_order", values="RET_1")
NLAGS = 20
acfs = []
for col in piv.columns:
    s = piv[col].dropna().values
    if len(s) > NLAGS + 5:
        acfs.append(acf(s, NLAGS))
mean_acf = np.nanmean(np.vstack(acfs), axis=0)

fig = px.bar(x=list(range(NLAGS + 1)), y=mean_acf,
             labels={"x": "lag (days)", "y": "mean ACF"},
             title=f"Average per-allocation return autocorrelation (n={len(acfs)} allocs)")
fig.add_hline(y=0, line_color="black")
fig.show()

In [ ]:
chk = df.sort_values(["alloc_order", "ts_order"]).copy()
chk["next_ts"]   = chk.groupby("alloc_order")["ts_order"].shift(-1)
chk["next_ret1"] = chk.groupby("alloc_order")["RET_1"].shift(-1)
consec = chk[chk["next_ts"] == chk["ts_order"] + 1].dropna(subset=["next_ret1"])
corr = np.corrcoef(consec["target"], consec["next_ret1"])[0, 1]
print(f"corr(target_t, RET_1_(t+1)) on consecutive days = {corr:.4f}  (n={len(consec):,})")
px.scatter(consec.sample(min(20000, len(consec))), x="target", y="next_ret1", opacity=0.3,
           title="target_t  vs  RET_1 of next day").show()

In [ ]:
mkt = df.groupby("ts_order")["target"].mean().rename("mkt")
df = df.merge(mkt, on="ts_order", how="left")
resid_var = (df["target"] - df["mkt"]).var()
share = 1 - resid_var / df["target"].var()
print(f"variance share explained by daily common factor ≈ {share:.3f}")
px.line(mkt, title="Daily cross-sectional mean return (the 'market' you cannot predict)").show()

In [ ]:
corr = df[RET_COLS + VOL_COLS + ["MEDIAN_DAILY_TURNOVER"]].corr()
px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
          title="Feature correlation (expect strong blocks within RET and within VOLUME)").show()

In [ ]:
Xt = pd.read_csv(x_test_path)
check_cols = ["RET_1", "RET_5", "RET_20", "SIGNED_VOLUME_1", "MEDIAN_DAILY_TURNOVER"]
fig = make_subplots(rows=1, cols=len(check_cols), subplot_titles=check_cols)
for j, c in enumerate(check_cols, start=1):
    lo, hi = df[c].quantile([0.01, 0.99])
    fig.add_trace(go.Histogram(x=df[c].clip(lo, hi), histnorm="probability density",
                               name="train", marker_color="blue", opacity=0.5, showlegend=(j==1)), row=1, col=j)
    fig.add_trace(go.Histogram(x=Xt[c].clip(lo, hi), histnorm="probability density",
                               name="test", marker_color="red", opacity=0.5, showlegend=(j==1)), row=1, col=j)
fig.update_layout(barmode="overlay", height=350, title="Train vs Test feature distributions")
fig.show()

In [ ]:
# Sign agreement between recent signed volume and recent return:
# +ve product => volume landed on the side that was moving in your favor ("confirmation")
df["sv1_x_ret1"] = np.sign(df["SIGNED_VOLUME_1"]) * np.sign(df["RET_1"])

agree = df.groupby(df["sv1_x_ret1"])["target_sign"].agg(["mean", "size"])
print(agree)   # is P(target>0) different on 'confirmation' vs 'contradiction' days?

# Bucket by signed-volume magnitude regime, see if momentum/reversal flips
df["sv_mag_bucket"] = pd.qcut(df["SIGNED_VOLUME_1"].abs(), 5, labels=False, duplicates="drop")
tab = df.groupby("sv_mag_bucket").apply(
    lambda d: pd.Series({
        "ret1_sign_agree_with_target": (np.sign(d["RET_1"]) == np.sign(d["target"])).mean(),
        "n": len(d),
    })
)
print(tab)

In [ ]:
import plotly.express as px
sub = df[["SIGNED_VOLUME_1", "target"]].dropna()
print("corr(SIGNED_VOLUME_1, target):", np.corrcoef(sub["SIGNED_VOLUME_1"], sub["target"])[0,1].round(4))

df["sv1_bucket"] = pd.qcut(df["SIGNED_VOLUME_1"], 10, labels=False, duplicates="drop")
by_bucket = df.groupby("sv1_bucket")["target_sign"].mean()
px.bar(by_bucket, title="P(target>0) across SIGNED_VOLUME_1 deciles").add_hline(y=0.5).show()